**Mosaic AI**

In [0]:
#1 Install Hugging Face Transformers and Torch
%pip install transformers torch

In [0]:
%restart_python

In [0]:
import pandas as pd
from pyspark.sql.functions import col

#1 Get Top 5 Products

top_products = spark.table("dev.ecommerce_governed.gold_events")\
                .orderBy(col("revenue").desc())\
                .limit(5)\
                .select("product_id","brand","revenue")\
                .toPandas()

#2. Simulate User Reviews
synthetic_reviews = [
    #Positive Reviews
    {"product_id": top_products.iloc[0]['product_id'], "review_text": "Absolutely love this phone! Fast and battery lasts forever."},
    {"product_id": top_products.iloc[1]['product_id'], "review_text": "Great value for the price. Best investment I made this year."},
    # Negative/Neutral Reviews
    {"product_id": top_products.iloc[2]['product_id'], "review_text": "It's okay, but the screen cracked way too easily. Disappointed."},
    {"product_id": top_products.iloc[3]['product_id'], "review_text": "Terrible delivery experience and the item arrived damaged."},
    {"product_id": top_products.iloc[4]['product_id'], "review_text": "Works as advertised, but a bit expensive for what you get."}
]

df_reviews = pd.DataFrame(synthetic_reviews)

display(df_reviews)


**Running the NLP Model**

In [0]:
from transformers import pipeline
import mlflow

sentiment_pipeline = pipeline("sentiment-analysis")

mlflow.set_experiment("/Users/rainbow.mem5@gmail.com/Day14-NLP-Analysis")

with mlflow.start_run(run_name="Customer_Voice_analysis"):
  mlflow.log_param("model_arch","distilbert-base-uncased-finetuned-sst-2-english")
  results = []

  for index, row in df_reviews.iterrows():
    prediction = sentiment_pipeline(row['review_text'])[0]
    results.append({
      "product_id": row['product_id'],
      "review_text": row['review_text'],
      "sentiment": prediction['label'],
      "score": prediction['score']
    })

    avg_conf = sum(r['score'] for r in results) / len(results)
    mlflow.log_metric("avg_confidence", avg_conf)
  
  df_sentiments = pd.DataFrame(results)

  display(df_sentiments)


In [0]:
df_final_insight = pd.merge(df_sentiments, top_products, on= "product_id")

spark_insight = spark.createDataFrame(df_final_insight)

display(spark_insight.select("brand","revenue","score","sentiment"))